In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.agents.cart_reinforce import ReinforceCartAgent
from src.plots import plot_durations


In [ ]:
env = gym.make("CartPole-v1", render_mode=None)

In [ ]:
LR = 3e-4
num_episodes = 1000

In [ ]:
agent = ReinforceCartAgent(env=env, learning_rate=LR)

# Training loop

In [ ]:
episode_durations = []

for _ in range(num_episodes):
    state, _ = env.reset()

    done = False

    probs = []
    rewards = []

    t = 0
    while not done:
        action, prob = agent.get_action(state)
        observation, reward, terminated, truncated, _ = env.step(action)
        
        probs.append(prob)
        rewards.append(reward)
        
        state = observation
        done = terminated or truncated
        t += 1
    
    agent.update(rewards, probs)

    episode_durations.append(t)
    plot_durations(episode_durations)

print('Complete')
plot_durations(episode_durations, show_result=True)
plt.ioff()
plt.show()

# Evaluation

In [ ]:
import numpy as np

test_env = gym.make("CartPole-v1", render_mode=None)
test_agent = ReinforceCartAgent(env, 0, 0)

test_agent.policy.load_state_dict(agent.policy.state_dict())

test_reward = []

for _ in tqdm(range(1000)):
    state, _ = test_env.reset()
    done = False
    total_reward = 0
    while not done:
        action = test_agent.act(state)
        next_state, reward, terminated, truncated, _ = test_env.step(action)

        done = terminated or truncated
        state = next_state
        total_reward += reward

    test_reward.append(total_reward)

print("avg: ", np.average(test_reward))

# Save if good

In [ ]:
from safetensors.torch import save_file

if np.average(test_reward) > 475.0:
    save_file(agent.policy.state_dict(), f"./data/reinforce_e{num_episodes}.safetensors")
